In [12]:
import pandas as pd

# Carrega admissões e desligamentos gerais por município e ocupação
emprego_geral_cbo = pd.read_parquet(
    "../data/processed/trabalho/emprego_geral_municipio_cbo_2025.parquet"
)

# Verifica as dimensões e os nomes das colunas
print("Dimensões:", emprego_geral_cbo.shape)
print("Colunas:", emprego_geral_cbo.columns.tolist())

# Visualiza os primeiros registros
display(emprego_geral_cbo.head())

Dimensões: (994623, 6)
Colunas: ['uf', 'cbo2002ocupação', 'admissoes', 'desligamentos', 'saldo', 'codigo_ibge']


,uf,cbo2002ocupação,admissoes,desligamentos,saldo,codigo_ibge
0,11,122520,1,0,1,1100015
1,11,123105,2,2,0,1100015
2,11,141115,3,1,2,1100015
3,11,141205,1,3,-2,1100015
4,11,141410,1,0,1,1100015


In [13]:
# Carrega os dados de migrantes por município e ocupação
emprego_migrantes_cbo = pd.read_parquet(
    "../data/processed/trabalho/emprego_migrantes_municipio_cbo_2025.parquet"
)

# Padroniza os códigos CBO nas duas tabelas
emprego_geral_cbo["cbo2002ocupação"] = (
    emprego_geral_cbo["cbo2002ocupação"].astype("string").str.zfill(6)
)

emprego_migrantes_cbo["cbo2002ocupacao"] = (
    emprego_migrantes_cbo["cbo2002ocupacao"].astype("string").str.zfill(6)
)

# Diferencia os indicadores gerais e os de migrantes
emprego_geral_cbo = emprego_geral_cbo.rename(columns={
    "admissoes": "admissoes_gerais",
    "desligamentos": "desligamentos_gerais",
    "saldo": "saldo_geral"
})

emprego_migrantes_cbo = emprego_migrantes_cbo.rename(columns={
    "cbo2002ocupacao": "cbo",
    "admissoes": "admissoes_migrantes",
    "desligamentos": "desligamentos_migrantes",
    "saldo": "saldo_migrantes"
})

emprego_geral_cbo = emprego_geral_cbo.rename(
    columns={"cbo2002ocupação": "cbo"}
)

# Integra as duas bases por município e ocupação
comparacao_cbo_2025 = emprego_geral_cbo.merge(
    emprego_migrantes_cbo,
    on=["codigo_ibge", "cbo"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Verifica a cobertura e a integridade
print("Dimensões:", comparacao_cbo_2025.shape)
print("Códigos duplicados:",
      comparacao_cbo_2025.duplicated(["codigo_ibge", "cbo"]).sum())

display(comparacao_cbo_2025["_merge"].value_counts())

Dimensões: (994623, 10)
Códigos duplicados: 0


_merge
left_only     943870
both           50753
right_only         0
Name: count, dtype: int64

In [14]:
# Seleciona combinações presentes nas duas bases
cbo_comparaveis = comparacao_cbo_2025[
    comparacao_cbo_2025["_merge"] == "both"
].copy()

# Verifica se as admissões de migrantes superam as gerais
inconsistencias = cbo_comparaveis[
    cbo_comparaveis["admissoes_migrantes"]
    > cbo_comparaveis["admissoes_gerais"]
]

# Resume a validação
print("Combinações comparadas:", len(cbo_comparaveis))
print("Combinações inconsistentes:", len(inconsistencias))

# Confere os totais de migrantes após a integração
print("Admissões migrantes:",
      cbo_comparaveis["admissoes_migrantes"].sum())

print("Desligamentos migrantes:",
      cbo_comparaveis["desligamentos_migrantes"].sum())

# Mostra eventuais inconsistências
display(
    inconsistencias[
        ["codigo_ibge", "cbo",
         "admissoes_migrantes", "admissoes_gerais"]
    ].head(20)
)

Combinações comparadas: 50753
Combinações inconsistentes: 0
Admissões migrantes: 393465.0
Desligamentos migrantes: 315556.0


,codigo_ibge,cbo,admissoes_migrantes,admissoes_gerais


In [15]:
# Agrupa as admissões gerais e de migrantes por ocupação CBO
ranking_comparativo_cbo = (
    comparacao_cbo_2025
    .groupby("cbo", dropna=False)
    .agg(
        admissoes_gerais=("admissoes_gerais", "sum"),
        admissoes_migrantes=("admissoes_migrantes", "sum")
    )
    .reset_index()
)

# Calcula a participação de migrantes nas admissões de cada ocupação
ranking_comparativo_cbo["participacao_migrante_pct"] = (
    ranking_comparativo_cbo["admissoes_migrantes"]
    / ranking_comparativo_cbo["admissoes_gerais"] * 100
).round(2)

# Carrega o dicionário oficial de ocupações CBO
ocupacoes_cbo = pd.read_csv(
    "../data/raw/trabalho/cbo/cbo2002-ocupacao.csv",
    sep=";",
    encoding="latin-1",
    dtype={"CODIGO": "string"}
)

# Padroniza os códigos e associa os títulos das ocupações
ocupacoes_cbo["CODIGO"] = ocupacoes_cbo["CODIGO"].str.zfill(6)

ranking_comparativo_cbo = ranking_comparativo_cbo.merge(
    ocupacoes_cbo[["CODIGO", "TITULO"]],
    left_on="cbo",
    right_on="CODIGO",
    how="left",
    validate="many_to_one"
)

# Exibe as 20 ocupações com mais admissões de migrantes
display(
    ranking_comparativo_cbo.nlargest(20, "admissoes_migrantes")[
        ["cbo", "TITULO", "admissoes_gerais",
         "admissoes_migrantes", "participacao_migrante_pct"]
    ]
)

,cbo,TITULO,admissoes_gerais,admissoes_migrantes,participacao_migrante_pct
2305,784205,Alimentador de linha de produção,1320551,52750.0,3.99
1413,514320,Faxineiro,1354871,26620.0,1.96
1397,513505,Auxiliar nos serviços de alimentação,519614,17550.0,3.38
2539,848520,Magarefe,66867,17010.0,25.44
1812,717020,Servente de obras,841361,15787.0,1.88
1519,521125,Repositor de mercadorias,555972,14440.0,2.60
1319,421125,Operador de caixa,779639,14198.0,1.82
1522,521140,Atendente de lojas e mercados,756910,13913.0,1.84
1586,622020,Trabalhador volante da agricultura,233908,10889.0,4.66
1395,513435,Atendente de lanchonete,439789,8696.0,1.98


In [16]:
# Calcula a participação migrante no conjunto das admissões
participacao_referencia = (
    ranking_comparativo_cbo["admissoes_migrantes"].sum()
    / ranking_comparativo_cbo["admissoes_gerais"].sum() * 100
)

# Mede a representação migrante relativa à referência geral
ranking_comparativo_cbo["indice_representacao"] = (
    ranking_comparativo_cbo["participacao_migrante_pct"]
    / participacao_referencia
)

# Exibe as ocupações com mais admissões migrantes
print(f"Participação de referência: {participacao_referencia:.2f}%")

display(
    ranking_comparativo_cbo.nlargest(20, "admissoes_migrantes")[
        ["TITULO", "admissoes_gerais", "admissoes_migrantes",
         "participacao_migrante_pct", "indice_representacao"]
    ]
)

Participação de referência: 1.50%


,TITULO,admissoes_gerais,admissoes_migrantes,participacao_migrante_pct,indice_representacao
2305,Alimentador de linha de produção,1320551,52750.0,3.99,2.668851
1413,Faxineiro,1354871,26620.0,1.96,1.311015
1397,Auxiliar nos serviços de alimentação,519614,17550.0,3.38,2.260831
2539,Magarefe,66867,17010.0,25.44,17.016435
1812,Servente de obras,841361,15787.0,1.88,1.257504
1519,Repositor de mercadorias,555972,14440.0,2.60,1.739101
1319,Operador de caixa,779639,14198.0,1.82,1.217371
1522,Atendente de lojas e mercados,756910,13913.0,1.84,1.230748
1586,Trabalhador volante da agricultura,233908,10889.0,4.66,3.117004
1395,Atendente de lanchonete,439789,8696.0,1.98,1.324392


In [17]:
# Seleciona ocupações com volume mínimo de admissões
oportunidades_municipais_cbo = comparacao_cbo_2025[
    (comparacao_cbo_2025["admissoes_gerais"] >= 100) &
    (comparacao_cbo_2025["admissoes_migrantes"] >= 30)
].copy()

# Calcula a participação migrante em cada município e ocupação
oportunidades_municipais_cbo["participacao_migrante_pct"] = (
    oportunidades_municipais_cbo["admissoes_migrantes"]
    / oportunidades_municipais_cbo["admissoes_gerais"] * 100
).round(2)

# Adiciona o nome das ocupações
oportunidades_municipais_cbo = oportunidades_municipais_cbo.merge(
    ocupacoes_cbo[["CODIGO", "TITULO"]],
    left_on="cbo",
    right_on="CODIGO",
    how="left",
    validate="many_to_one"
)

# Exibe as maiores participações migrantes
display(
    oportunidades_municipais_cbo.nlargest(
        20, "participacao_migrante_pct"
    )[
        ["codigo_ibge", "TITULO", "admissoes_gerais",
         "admissoes_migrantes", "participacao_migrante_pct"]
    ]
)

,codigo_ibge,TITULO,admissoes_gerais,admissoes_migrantes,participacao_migrante_pct
654,4107207,Abatedor,295,273.0,92.54
435,3550308,Abatedor,991,845.0,85.27
1442,4310439,Trabalhador no cultivo de árvores frutíferas,333,246.0,73.87
1405,4307005,Magarefe,1056,780.0,73.86
1619,5005707,Trabalhador polivalente do curtimento de couro...,101,74.0,73.27
1359,4305108,Auxiliar de produção farmacêutica,215,155.0,72.09
1173,4209300,Trabalhador no cultivo de árvores frutíferas,786,565.0,71.88
1059,4206652,Magarefe,2284,1569.0,68.70
1418,4308201,Trabalhador volante da agricultura,2077,1415.0,68.13
979,4204202,Magarefe,5470,3653.0,66.78


In [18]:
# Carrega os nomes dos municípios já padronizados
municipios = pd.read_parquet(
    "../data/processed/indicadores_completos_municipais.parquet"
)[["codigo_ibge", "municipio", "uf"]].drop_duplicates("codigo_ibge")

In [19]:
# Acrescenta município e UF às oportunidades por ocupação

oportunidades_municipais_cbo = oportunidades_municipais_cbo.merge(
    municipios,
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

In [20]:
# Remove a UF duplicada e padroniza o nome da coluna
oportunidades_municipais_cbo = oportunidades_municipais_cbo.drop(
    columns=["uf_y"]
).rename(columns={"uf_x": "uf"})

# Exibe o Top 20 com município e UF
display(
    oportunidades_municipais_cbo.nlargest(
        20, "participacao_migrante_pct"
    )[
        ["municipio", "uf", "TITULO",
         "admissoes_gerais", "admissoes_migrantes",
         "participacao_migrante_pct"]
    ]
)

,municipio,uf,TITULO,admissoes_gerais,admissoes_migrantes,participacao_migrante_pct
654,DOIS VIZINHOS,41,Abatedor,295,273.0,92.54
435,SÃO PAULO,35,Abatedor,991,845.0,85.27
1442,IPÊ,43,Trabalhador no cultivo de árvores frutíferas,333,246.0,73.87
1405,ERECHIM,43,Magarefe,1056,780.0,73.86
1619,NAVIRAÍ,50,Trabalhador polivalente do curtimento de couro...,101,74.0,73.27
1359,CAXIAS DO SUL,43,Auxiliar de produção farmacêutica,215,155.0,72.09
1173,LAGES,42,Trabalhador no cultivo de árvores frutíferas,786,565.0,71.88
1059,GUATAMBÚ,42,Magarefe,2284,1569.0,68.70
1418,FLORES DA CUNHA,43,Trabalhador volante da agricultura,2077,1415.0,68.13
979,CHAPECÓ,42,Magarefe,5470,3653.0,66.78


In [21]:
# Carrega a oferta de cursos técnicos por município
oferta_cursos = pd.read_parquet(
    "../data/processed/educacao/educacao_municipio_curso_2025.parquet"
)

# Verifica a estrutura e alguns exemplos
print("Dimensões:", oferta_cursos.shape)
print("Colunas:", oferta_cursos.columns.tolist())

display(oferta_cursos.head())

Dimensões: (18793, 6)
Colunas: ['codigo_ibge', 'CO_CURSO_EDUC_PROFISSIONAL', 'NO_CURSO_EDUC_PROFISSIONAL', 'NO_AREA_CURSO_PROFISSIONAL', 'instituicoes', 'matriculas']


,codigo_ibge,CO_CURSO_EDUC_PROFISSIONAL,NO_CURSO_EDUC_PROFISSIONAL,NO_AREA_CURSO_PROFISSIONAL,instituicoes,matriculas
0,1100023,1002,Análises clínicas,Ambiente e saúde,1,29
1,1100023,1006,Enfermagem,Ambiente e saúde,1,254
2,1100023,1023,Radiologia,Ambiente e saúde,1,21
3,1100023,1999,Outros - Eixo Ambiente e Saúde,Ambiente e saúde,1,71
4,1100023,3040,Eletrotécnica,Controle e processos industriais,1,53


In [22]:
# Resume a oferta de cursos por área de formação
resumo_areas = (
    oferta_cursos
    .groupby("NO_AREA_CURSO_PROFISSIONAL")
    .agg(
        cursos_distintos=("CO_CURSO_EDUC_PROFISSIONAL", "nunique"),
        municipios=("codigo_ibge", "nunique"),
        matriculas=("matriculas", "sum")
    )
    .sort_values("matriculas", ascending=False)
)

display(resumo_areas)

,cursos_distintos,municipios,matriculas
NO_AREA_CURSO_PROFISSIONAL,,,
Gestão e negócios,18,2287,711071
Ambiente e saúde,29,1383,503998
Informação e comunicação,9,1719,424627
Controle e processos industriais,26,674,292384
Recursos naturais,15,1349,169319
Segurança,4,633,99234
Produção industrial,20,394,62803
Infraestrutura,15,335,53081
Produção cultural e design,29,255,52711
